# Runnable Event Streaming

The `event_stream.py` module contains the internal tracing machinery that powers the asynchronous Runnable event-stream API.

It converts callback activity into `StandardStreamEvent` and `CustomStreamEvent` records, preserves parent-run lineage, filters events by name, type, and tag, and taps streamed Runnable output without changing the values delivered to the caller.

# RunInfo

`RunInfo` stores the metadata required to create streaming events for one active Runnable run.

## Bases

- `TypedDict`

## Attributes

1. `name`: Stores the display name of the run.
   * **Type:**
     ```python
     name: str
     ```

2. `tags`: Stores the tags associated with the run.
   * **Type:**
     ```python
     tags: list[str]
     ```

3. `metadata`: Stores the metadata associated with the run.
   * **Type:**
     ```python
     metadata: dict[str, Any]
     ```

4. `run_type`: Stores the Runnable category used to construct lifecycle event names.
   * **Type:**
     ```python
     run_type: str
     ```

5. `inputs`: Optionally stores the input associated with the run.
   * **Type:**
     ```python
     inputs: NotRequired[Any]
     ```

6. `parent_run_id`: Stores the identifier of the immediate parent run.
   * **Type:**
     ```python
     parent_run_id: UUID | None
     ```

7. `tool_call_id`: Optionally stores the tool-call identifier associated with a tool run.
   * **Type:**
     ```python
     tool_call_id: NotRequired[
         str | None
     ]
     ```

# _AstreamEventsCallbackHandler

`_AstreamEventsCallbackHandler` is the internal asynchronous callback handler used by the version 2 event-stream implementation.

It receives Runnable lifecycle callbacks, converts them into stream-event dictionaries, filters those events, and sends accepted events through an in-memory asynchronous stream.

Although this class is internal, it is the primary event-generation component in this module.

## Bases

- `AsyncCallbackHandler`
- `_StreamingCallbackHandler[Any]`

## Attributes

1. `run_map`: Maps active run identifiers to their `RunInfo` records.

   An entry is removed when its corresponding run ends.

   * **Type:**
     ```python
     run_map: dict[
         UUID,
         RunInfo
     ]
     ```

2. `parent_map`: Maps run identifiers to immediate parent-run identifiers.

   This mapping is kept separately because a parent end callback may occur before a child end callback.

   * **Type:**
     ```python
     parent_map: dict[
         UUID,
         UUID | None
     ]
     ```

3. `is_tapped`: Tracks runs whose output iterators have already been tapped.

   This prevents duplicate stream events when more than one layer attempts to observe the same output.

   * **Type:**
     ```python
     is_tapped: dict[
         UUID,
         Any
     ]
     ```

4. `root_event_filter`: Stores the include and exclude rules used to decide which events enter the output stream.
   * **Type:**
     ```python
     root_event_filter: _RootEventFilter
     ```

5. `send_stream`: Stores the sending side of the internal memory stream.
   * **Type:**
     ```python
     send_stream: Any
     ```

6. `receive_stream`: Stores the receiving side of the internal memory stream.
   * **Type:**
     ```python
     receive_stream: Any
     ```

### Methods

1. `__init__`: Initializes the callback handler, filtering rules, run maps, and in-memory event stream.

   Each include filter restricts output to matching events. Each exclude filter removes matching events. When no active event loop is available, a new loop is created for the memory stream.

   * **Syntax:**
     ```python
     __init__(
         self,
         *args: Any, # Positional arguments passed to the callback-handler base
         include_names: Sequence[str] | None = None, # Included Runnable names
         include_types: Sequence[str] | None = None, # Included Runnable types
         include_tags: Sequence[str] | None = None, # Included Runnable tags
         exclude_names: Sequence[str] | None = None, # Excluded Runnable names
         exclude_types: Sequence[str] | None = None, # Excluded Runnable types
         exclude_tags: Sequence[str] | None = None, # Excluded Runnable tags
         **kwargs: Any # Additional callback-handler arguments
     ) -> None
     ```

2. `__aiter__`: Returns an asynchronous iterator over events received from the internal memory stream.
   * **Syntax:**
     ```python
     __aiter__(
         self
     ) -> AsyncIterator[Any]
     ```

3. `tap_output_aiter`: Observes an asynchronous output iterator and emits one stream event for each chunk.

   The original chunks are yielded unchanged. Only the first tap for a run emits events; later taps pass the chunks through without duplicating them.

   When the run has already ended, the first available chunk is yielded without creating an event.

   * **Syntax:**
     ```python
     async tap_output_aiter(
         self,
         run_id: UUID, # Identifier of the run producing the output
         output: AsyncIterator[T] # Asynchronous output iterator to observe
     ) -> AsyncIterator[T]
     ```

4. `tap_output_iter`: Observes a synchronous output iterator and emits one stream event for each chunk.

   The original chunks are yielded unchanged. Only the first tap for a run emits events; later taps pass the chunks through without duplicating them.

   * **Syntax:**
     ```python
     tap_output_iter(
         self,
         run_id: UUID, # Identifier of the run producing the output
         output: Iterator[T] # Synchronous output iterator to observe
     ) -> Iterator[T]
     ```

5. `on_chat_model_start`: Records a chat-model run and emits an `on_chat_model_start` event.

   The event input contains the supplied message batches.

   * **Syntax:**
     ```python
     async on_chat_model_start(
         self,
         serialized: dict[str, Any], # Serialized chat-model information
         messages: list[
             list[BaseMessage]
         ], # Message batches supplied to the chat model
         *,
         run_id: UUID, # Unique run identifier
         tags: list[str] | None = None, # Optional run tags
         parent_run_id: UUID | None = None, # Optional parent-run identifier
         metadata: dict[str, Any] | None = None, # Optional run metadata
         name: str | None = None, # Optional run name
         **kwargs: Any # Additional callback arguments
     ) -> None
     ```

6. `on_llm_start`: Records a text-completion model run and emits an `on_llm_start` event.

   The event input contains the supplied prompts.

   * **Syntax:**
     ```python
     async on_llm_start(
         self,
         serialized: dict[str, Any], # Serialized language-model information
         prompts: list[str], # Prompts supplied to the language model
         *,
         run_id: UUID, # Unique run identifier
         tags: list[str] | None = None, # Optional run tags
         parent_run_id: UUID | None = None, # Optional parent-run identifier
         metadata: dict[str, Any] | None = None, # Optional run metadata
         name: str | None = None, # Optional run name
         **kwargs: Any # Additional callback arguments
     ) -> None
     ```

7. `on_custom_event`: Emits a user-defined `on_custom_event` event.

   The supplied name is used both as the event's custom name and as the event type passed to the filter.

   * **Syntax:**
     ```python
     async on_custom_event(
         self,
         name: str, # Name of the custom event
         data: Any, # Custom event payload
         *,
         run_id: UUID, # Run associated with the event
         tags: list[str] | None = None, # Optional event tags
         metadata: dict[str, Any] | None = None, # Optional event metadata
         **kwargs: Any # Additional callback arguments
     ) -> None
     ```

8. `on_llm_new_token`: Emits a streaming event for an LLM or chat-model token.

   Chat-model tokens are represented as message chunks. Text-completion model tokens are represented as generation chunks.

   No event is emitted when the run's output has already been tapped. An `AssertionError` is raised when the run is unknown, and a `ValueError` is raised when its type is neither `"llm"` nor `"chat_model"`.

   * **Syntax:**
     ```python
     async on_llm_new_token(
         self,
         token: str
         | list[
             str
             | dict[str, Any]
         ], # Token or structured content blocks
         *,
         chunk: GenerationChunk
         | ChatGenerationChunk
         | None = None, # Optional generation chunk
         run_id: UUID, # Run receiving the token
         parent_run_id: UUID | None = None, # Compatibility parent-run identifier
         **kwargs: Any # Additional callback arguments
     ) -> None
     ```

9. `on_llm_end`: Completes an LLM or chat-model run and emits its end event.

   Chat-model output is reduced to the first available generated message. Text-completion output contains serialized generation information and the model-level output dictionary.

   A `ValueError` is raised when the stored run type is unexpected.

   * **Syntax:**
     ```python
     async on_llm_end(
         self,
         response: LLMResult, # Completed language-model result
         *,
         run_id: UUID, # Run to complete
         **kwargs: Any # Additional callback arguments
     ) -> None
     ```

10. `on_chain_start`: Records a chain or custom Runnable run and emits its start event.

    The supplied `run_type` determines the event prefix and defaults to `"chain"`. The placeholder input `{"input": ""}` is omitted from the start-event data.

    * **Syntax:**
      ```python
      async on_chain_start(
          self,
          serialized: dict[str, Any], # Serialized Runnable information
          inputs: dict[str, Any], # Runnable inputs
          *,
          run_id: UUID, # Unique run identifier
          tags: list[str] | None = None, # Optional run tags
          parent_run_id: UUID | None = None, # Optional parent-run identifier
          metadata: dict[str, Any] | None = None, # Optional run metadata
          run_type: str | None = None, # Optional specialized Runnable type
          name: str | None = None, # Optional run name
          **kwargs: Any # Additional callback arguments
      ) -> None
      ```

11. `on_chain_end`: Completes a chain or custom Runnable run and emits its end event.

    Inputs supplied at completion take priority over inputs stored at start.

    * **Syntax:**
      ```python
      async on_chain_end(
          self,
          outputs: dict[str, Any], # Runnable outputs
          *,
          run_id: UUID, # Run to complete
          inputs: dict[str, Any] | None = None, # Optional completed input value
          **kwargs: Any # Additional callback arguments
      ) -> None
      ```

12. `on_tool_start`: Records a tool run and emits an `on_tool_start` event.

    Structured tool input and an optional tool-call identifier are retained for the corresponding end or error event.

    * **Syntax:**
      ```python
      async on_tool_start(
          self,
          serialized: dict[str, Any], # Serialized tool information
          input_str: str, # String representation of the tool input
          *,
          run_id: UUID, # Unique run identifier
          tags: list[str] | None = None, # Optional run tags
          parent_run_id: UUID | None = None, # Optional parent-run identifier
          metadata: dict[str, Any] | None = None, # Optional run metadata
          name: str | None = None, # Optional run name
          inputs: dict[str, Any] | None = None, # Optional structured tool input
          **kwargs: Any # Additional callback arguments
      ) -> None
      ```

13. `on_tool_error`: Completes a tool run with an error and emits an `on_tool_error` event.

    The event data contains the error, original structured input, and tool-call identifier. An `AssertionError` is raised when the run has no stored input entry.

    * **Syntax:**
      ```python
      async on_tool_error(
          self,
          error: BaseException, # Error raised during tool execution
          *,
          run_id: UUID, # Run to complete
          parent_run_id: UUID | None = None, # Compatibility parent-run identifier
          tags: list[str] | None = None, # Compatibility tags
          **kwargs: Any # Additional callback arguments
      ) -> None
      ```

14. `on_tool_end`: Completes a tool run and emits an `on_tool_end` event.

    The event data contains both the tool output and its original structured input.

    * **Syntax:**
      ```python
      async on_tool_end(
          self,
          output: Any, # Tool output
          *,
          run_id: UUID, # Run to complete
          **kwargs: Any # Additional callback arguments
      ) -> None
      ```

15. `on_retriever_start`: Records a retriever run and emits an `on_retriever_start` event.

    The retrieval query is stored under the `"query"` input key.

    * **Syntax:**
      ```python
      async on_retriever_start(
          self,
          serialized: dict[str, Any], # Serialized retriever information
          query: str, # Retrieval query
          *,
          run_id: UUID, # Unique run identifier
          parent_run_id: UUID | None = None, # Optional parent-run identifier
          tags: list[str] | None = None, # Optional run tags
          metadata: dict[str, Any] | None = None, # Optional run metadata
          name: str | None = None, # Optional run name
          **kwargs: Any # Additional callback arguments
      ) -> None
      ```

16. `on_retriever_end`: Completes a retriever run and emits an `on_retriever_end` event.

    The event data contains the retrieved documents and the original query input.

    * **Syntax:**
      ```python
      async on_retriever_end(
          self,
          documents: Sequence[Document], # Retrieved documents
          *,
          run_id: UUID, # Run to complete
          **kwargs: Any # Additional callback arguments
      ) -> None
      ```

17. `__deepcopy__`: Returns the same callback-handler instance rather than creating a deep copy.
    * **Syntax:**
      ```python
      __deepcopy__(
          self,
          memo: dict[int, Any] | None = None # Optional copy memo
      ) -> _AstreamEventsCallbackHandler
      ```

18. `__copy__`: Returns the same callback-handler instance rather than creating a shallow copy.
    * **Syntax:**
      ```python
      __copy__(
          self
      ) -> _AstreamEventsCallbackHandler
      ```

## Internal Functions

1. `_astream_events_implementation_v1`: Implements version 1 of the asynchronous event-stream API by converting streamed run-log patches into lifecycle events.

   It builds a `RunLog`, determines whether each log entry represents a start, stream, or end event, and emits corresponding `StandardStreamEvent` objects.

   Version 1 does not provide parent-run identifiers, so every emitted event uses an empty `parent_ids` sequence. Root-event filtering is applied independently to the root start, stream, and end events.

   An `AssertionError` is raised when a streamed-output entry unexpectedly contains a number of chunks other than one.

   * **Syntax:**
     ```python
     async _astream_events_implementation_v1(
         runnable: Runnable[
             Input,
             Output
         ], # Runnable whose events are streamed
         value: Any, # Input supplied to the Runnable
         config: RunnableConfig | None = None, # Runtime configuration
         *,
         include_names: Sequence[str] | None = None, # Included Runnable names
         include_types: Sequence[str] | None = None, # Included Runnable types
         include_tags: Sequence[str] | None = None, # Included Runnable tags
         exclude_names: Sequence[str] | None = None, # Excluded Runnable names
         exclude_types: Sequence[str] | None = None, # Excluded Runnable types
         exclude_tags: Sequence[str] | None = None, # Excluded Runnable tags
         **kwargs: Any # Additional streaming arguments
     ) -> AsyncIterator[
         StandardStreamEvent
     ]
     ```

2. `_astream_events_implementation_v2`: Implements version 2 of the asynchronous event-stream API using `_AstreamEventsCallbackHandler`.

   It assigns a UUIDv7 run identifier when the configuration has no `run_id`, inserts the event handler into the callback configuration, runs the Runnable's asynchronous stream in a separate task, and yields events from the handler's memory stream.

   The original input is inserted into the first emitted event. That input is removed from the corresponding root end event because it has already been supplied at the start.

   Existing callbacks may be absent, stored as a list, or stored in a `BaseCallbackManager`. A `ValueError` is raised for an unsupported callback container. Cancellation of the consumer also cancels and awaits the background Runnable task.

   * **Syntax:**
     ```python
     async _astream_events_implementation_v2(
         runnable: Runnable[
             Input,
             Output
         ], # Runnable whose events are streamed
         value: Any, # Input supplied to the Runnable
         config: RunnableConfig | None = None, # Runtime configuration
         *,
         include_names: Sequence[str] | None = None, # Included Runnable names
         include_types: Sequence[str] | None = None, # Included Runnable types
         include_tags: Sequence[str] | None = None, # Included Runnable tags
         exclude_names: Sequence[str] | None = None, # Excluded Runnable names
         exclude_types: Sequence[str] | None = None, # Excluded Runnable types
         exclude_tags: Sequence[str] | None = None, # Excluded Runnable tags
         **kwargs: Any # Additional streaming arguments
     ) -> AsyncIterator[
         StandardStreamEvent
     ]
     ```